### 1. OASIS Dataset - Download and Preprocess

#### 1A - Download data from kaggle and split for ReID tasks
Please create a **kaggle account** and prepare to enter your *USERNAME* and *KEY* to be able to downlaod the data...

In [9]:
# =============================================================================
# OASIS Brain MRI Dataset Preparation - Re-ID Task
# =============================================================================
# This notebook prepares the OASIS dataset for Re-Identification task.
#
# RE-ID TASK (Closed-Set, Slice-Level Split):
#    - All subjects appear in train/val/test
#    - Different slice indices per split (intentional)
#    - Train slices: [100, 108, 145, 153]
#    - Val slices: [115]
#    - Test slices: [122, 130, 138]
# =============================================================================

import os
import json
import glob
import pickle
from pathlib import Path
from typing import Dict, List, Tuple, Any
from tqdm import tqdm
import numpy as np
from PIL import Image

# =============================================================================
# CONFIGURATION
# =============================================================================

# Paths
PROJECT_ROOT = Path('.')
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'oasis' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'oasis' / 'processed'

# Kaggle dataset
KAGGLE_DATASET = "ninadaithal/imagesoasis"

# Image preprocessing
CROP_WIDTH = 248
IMAGE_SIZE = (224, 224)
SLICE_RANGE = (100, 160)

# Re-ID slice configuration (deterministic)
REID_TRAIN_SLICES = [100, 108, 145, 153]
REID_VAL_SLICES = [115]
REID_TEST_SLICES = [122, 130, 138]
ALL_SLICES = sorted(set(REID_TRAIN_SLICES + REID_VAL_SLICES + REID_TEST_SLICES))

print("Configuration:")
print(f"  Raw data:     {RAW_DATA_DIR}")
print(f"  Processed:    {PROCESSED_DIR}")
print(f"  Image size:   {IMAGE_SIZE}")
print(f"  Train slices: {REID_TRAIN_SLICES}")
print(f"  Val slices:   {REID_VAL_SLICES}")
print(f"  Test slices:  {REID_TEST_SLICES}")

Configuration:
  Raw data:     data/oasis/raw
  Processed:    data/oasis/processed
  Image size:   (224, 224)
  Train slices: [100, 108, 145, 153]
  Val slices:   [115]
  Test slices:  [122, 130, 138]


In [10]:
# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def save_pickle(data: Any, path: Path) -> None:
    """Save data to pickle file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'wb') as f:
        pickle.dump(data, f)
    print(f"  Saved: {path}")


def load_pickle(path: Path) -> Any:
    """Load data from pickle file."""
    with open(path, 'rb') as f:
        return pickle.load(f)


def load_and_preprocess_image(filepath: str) -> np.ndarray:
    """
    Load image, crop, resize to 224x224, and normalize to [0,1].
    """
    img = Image.open(filepath).convert('L')
    arr = np.array(img, dtype=np.float32)
    
    # Crop width
    arr = arr[:, :CROP_WIDTH]
    
    # Resize to 224x224
    img_cropped = Image.fromarray(arr)
    img_resized = img_cropped.resize(IMAGE_SIZE, Image.BILINEAR)
    arr_resized = np.array(img_resized, dtype=np.float32)
    
    # Normalize to [0, 1]
    arr_resized = arr_resized / 255.0
    
    return arr_resized


def extract_subject_id(filename: str) -> str:
    """Extract subject ID. E.g., 'OAS1_0001_MR1_mpr-1_100.jpg' -> 'OAS1_0001'"""
    parts = filename.split('_')
    return '_'.join(parts[:2]) if len(parts) >= 2 else None


def extract_slice_number(filename: str) -> int:
    """Extract slice number. E.g., 'OAS1_0001_MR1_mpr-1_100.jpg' -> 100"""
    try:
        return int(filename.split('_')[-1].split('.')[0])
    except (ValueError, IndexError):
        return None

In [11]:
# =============================================================================
# KAGGLE SETUP & DOWNLOAD
# =============================================================================

def setup_kaggle_credentials():
    """Prompt user for Kaggle credentials and configure environment."""
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_json = kaggle_dir / 'kaggle.json'
    
    # Check if already configured
    if kaggle_json.exists():
        print("✓ Kaggle credentials found at ~/.kaggle/kaggle.json")
        return True
    
    print("="*60)
    print("KAGGLE SETUP REQUIRED")
    print("="*60)
    print("To download the dataset, you need Kaggle API credentials.")
    print("Get them from: kaggle.com → Profile → Settings → API → Create New Token")
    print("="*60)
    
    username = input("Enter your Kaggle username: ").strip()
    api_key = input("Enter your Kaggle API key: ").strip()
    
    if not username or not api_key:
        raise ValueError("Username and API key are required.")
    
    # Create credentials file
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    with open(kaggle_json, 'w') as f:
        json.dump({"username": username, "key": api_key}, f)
    os.chmod(kaggle_json, 0o600)
    
    print(f"✓ Credentials saved to {kaggle_json}")
    return True


def download_oasis_dataset() -> Path:
    """Download OASIS dataset from Kaggle."""
    # Check if already downloaded
    if RAW_DATA_DIR.exists() and any(RAW_DATA_DIR.rglob('*.jpg')):
        print(f"✓ Dataset already exists at {RAW_DATA_DIR}")
        return RAW_DATA_DIR
    
    # Setup credentials
    setup_kaggle_credentials()
    
    # Create directory
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    
    # Download
    print(f"\n⬇️  Downloading {KAGGLE_DATASET}...")
    exit_code = os.system(f"kaggle datasets download -d {KAGGLE_DATASET} -p {RAW_DATA_DIR} --unzip")
    
    if exit_code != 0:
        raise RuntimeError("Download failed. Check your credentials and try again.")
    
    print("✓ Download complete")
    return RAW_DATA_DIR

In [12]:
# =============================================================================
# SCAN AND MAP FILES
# =============================================================================

def scan_oasis_files(data_dir: Path) -> Dict[str, Dict[int, str]]:
    """
    Scan directory and build subject -> slice -> filepath mapping.
    
    Returns:
        Dict: subject_id -> {slice_num: filepath}
    """
    print("\n[SCAN] Scanning image files...")
    
    all_files = list(data_dir.rglob('*.jpg'))
    if not all_files:
        raise FileNotFoundError(f"No .jpg files found in {data_dir}")
    
    print(f"  Found {len(all_files)} total files")
    
    subject_dict = {}
    slice_min, slice_max = SLICE_RANGE
    
    for filepath in tqdm(all_files, desc="  Mapping"):
        filename = filepath.name
        
        subject_id = extract_subject_id(filename)
        slice_num = extract_slice_number(filename)
        
        if subject_id is None or slice_num is None:
            continue
        
        if not (slice_min <= slice_num <= slice_max):
            continue
        
        if subject_id not in subject_dict:
            subject_dict[subject_id] = {}
        
        if slice_num not in subject_dict[subject_id]:
            subject_dict[subject_id][slice_num] = str(filepath)
    
    print(f"  ✓ Mapped {len(subject_dict)} subjects")
    return subject_dict


def filter_valid_subjects(subject_dict: Dict, required_slices: List[int]) -> Dict:
    """Filter to subjects having all required slices."""
    print(f"\n[FILTER] Requiring slices: {required_slices}")
    
    required_set = set(required_slices)
    valid = {sid: slices for sid, slices in subject_dict.items() 
             if required_set.issubset(set(slices.keys()))}
    
    print(f"  ✓ {len(valid)} subjects have all required slices")
    return valid

In [13]:
# =============================================================================
# PREPARE RE-ID DATASET
# =============================================================================

def prepare_reid_dataset(subject_dict: Dict, output_dir: Path) -> Tuple[List, List, List]:
    """
    Prepare Re-ID dataset with slice-level splitting.
    
    All subjects appear in all splits with different slices.
    """
    print("\n[RE-ID] Preparing dataset...")
    
    # Deterministic label mapping (sorted for reproducibility)
    sorted_subjects = sorted(subject_dict.keys())
    subject_to_label = {sid: idx for idx, sid in enumerate(sorted_subjects)}
    
    train_data, val_data, test_data = [], [], []
    
    for subject_id in tqdm(sorted_subjects, desc="  Processing"):
        slices = subject_dict[subject_id]
        reid_label = subject_to_label[subject_id]
        
        # Train slices
        for slice_idx in REID_TRAIN_SLICES:
            if slice_idx in slices:
                train_data.append({
                    'image': load_and_preprocess_image(slices[slice_idx]),
                    'reid_label': reid_label,
                    'subject_id': subject_id,
                    'slice_index': slice_idx,
                })
        
        # Val slices
        for slice_idx in REID_VAL_SLICES:
            if slice_idx in slices:
                val_data.append({
                    'image': load_and_preprocess_image(slices[slice_idx]),
                    'reid_label': reid_label,
                    'subject_id': subject_id,
                    'slice_index': slice_idx,
                })
        
        # Test slices
        for slice_idx in REID_TEST_SLICES:
            if slice_idx in slices:
                test_data.append({
                    'image': load_and_preprocess_image(slices[slice_idx]),
                    'reid_label': reid_label,
                    'subject_id': subject_id,
                    'slice_index': slice_idx,
                })
    
    # Save
    reid_dir = output_dir / 'reid'
    save_pickle(train_data, reid_dir / 'train.pkl')
    save_pickle(val_data, reid_dir / 'val.pkl')
    save_pickle(test_data, reid_dir / 'test.pkl')
    save_pickle(subject_to_label, reid_dir / 'subject_to_label.pkl')
    save_pickle({
        'train_slices': REID_TRAIN_SLICES,
        'val_slices': REID_VAL_SLICES,
        'test_slices': REID_TEST_SLICES,
        'num_subjects': len(subject_to_label),
        'image_size': IMAGE_SIZE,
    }, reid_dir / 'metadata.pkl')
    
    return train_data, val_data, test_data

In [14]:
# =============================================================================
# VALIDATION
# =============================================================================

def validate_reid_dataset(output_dir: Path) -> bool:
    """Verify Re-ID dataset integrity."""
    print("\n[VALIDATE] Checking dataset...")
    
    reid_dir = output_dir / 'reid'
    train = load_pickle(reid_dir / 'train.pkl')
    val = load_pickle(reid_dir / 'val.pkl')
    test = load_pickle(reid_dir / 'test.pkl')
    
    # Check 1: All subjects in all splits
    train_subs = {d['subject_id'] for d in train}
    val_subs = {d['subject_id'] for d in val}
    test_subs = {d['subject_id'] for d in test}
    
    if train_subs == val_subs == test_subs:
        print(f"  ✓ Closed-set: {len(train_subs)} subjects in all splits")
    else:
        print("  ✗ Subject mismatch across splits!")
        return False
    
    # Check 2: Slices separated
    train_slices = {d['slice_index'] for d in train}
    val_slices = {d['slice_index'] for d in val}
    test_slices = {d['slice_index'] for d in test}
    
    if train_slices == set(REID_TRAIN_SLICES) and \
       val_slices == set(REID_VAL_SLICES) and \
       test_slices == set(REID_TEST_SLICES):
        print(f"  ✓ Slice separation verified")
    else:
        print("  ✗ Slice mismatch!")
        return False
    
    # Check 3: Image shape
    if train[0]['image'].shape == IMAGE_SIZE:
        print(f"  ✓ Image shape: {IMAGE_SIZE}")
    else:
        print(f"  ✗ Wrong image shape: {train[0]['image'].shape}")
        return False
    
    # Summary
    print(f"\n  Train: {len(train)} samples")
    print(f"  Val:   {len(val)} samples")
    print(f"  Test:  {len(test)} samples")
    
    return True

In [15]:
# =============================================================================
# MAIN EXECUTION
# =============================================================================

# Step 1: Download
download_oasis_dataset()

# Step 2: Find data directory
data_dir = RAW_DATA_DIR / 'Data' if (RAW_DATA_DIR / 'Data').exists() else RAW_DATA_DIR

# Step 3: Scan files
subject_dict = scan_oasis_files(data_dir)

# Step 4: Filter valid subjects
valid_subjects = filter_valid_subjects(subject_dict, ALL_SLICES)

# Step 5: Prepare dataset
train_data, val_data, test_data = prepare_reid_dataset(valid_subjects, PROCESSED_DIR)

# Step 6: Validate
validate_reid_dataset(PROCESSED_DIR)

print("\n" + "="*60)
print("✓ OASIS RE-ID DATA PREPARATION COMPLETE")
print("="*60)
print(f"\nOutput: {PROCESSED_DIR}/reid/")
print(f"  ├── train.pkl  ({len(train_data)} samples, slices {REID_TRAIN_SLICES})")
print(f"  ├── val.pkl    ({len(val_data)} samples, slices {REID_VAL_SLICES})")
print(f"  ├── test.pkl   ({len(test_data)} samples, slices {REID_TEST_SLICES})")
print(f"  ├── subject_to_label.pkl")
print(f"  └── metadata.pkl")

KAGGLE SETUP REQUIRED
To download the dataset, you need Kaggle API credentials.
Get them from: kaggle.com → Profile → Settings → API → Create New Token


Enter your Kaggle username:  sehaay
Enter your Kaggle API key:  735a2076c4bf783483383a7ce8aae853


✓ Credentials saved to /home/jupyter/.kaggle/kaggle.json

⬇️  Downloading ninadaithal/imagesoasis...
Dataset URL: https://www.kaggle.com/datasets/ninadaithal/imagesoasis
License(s): apache-2.0


100%|██████████| 1.23G/1.23G [00:01<00:00, 1.06GB/s]



✓ Download complete

[SCAN] Scanning image files...
  Found 86437 total files


  Mapping: 100%|██████████| 86437/86437 [00:00<00:00, 443317.77it/s]


  ✓ Mapped 347 subjects

[FILTER] Requiring slices: [100, 108, 115, 122, 130, 138, 145, 153]
  ✓ 347 subjects have all required slices

[RE-ID] Preparing dataset...


  Processing: 100%|██████████| 347/347 [00:05<00:00, 64.18it/s]


  Saved: data/oasis/processed/reid/train.pkl
  Saved: data/oasis/processed/reid/val.pkl
  Saved: data/oasis/processed/reid/test.pkl
  Saved: data/oasis/processed/reid/subject_to_label.pkl
  Saved: data/oasis/processed/reid/metadata.pkl

[VALIDATE] Checking dataset...
  ✓ Closed-set: 347 subjects in all splits
  ✓ Slice separation verified
  ✓ Image shape: (224, 224)

  Train: 1388 samples
  Val:   347 samples
  Test:  1041 samples

✓ OASIS RE-ID DATA PREPARATION COMPLETE

Output: data/oasis/processed/reid/
  ├── train.pkl  (1388 samples, slices [100, 108, 145, 153])
  ├── val.pkl    (347 samples, slices [115])
  ├── test.pkl   (1041 samples, slices [122, 130, 138])
  ├── subject_to_label.pkl
  └── metadata.pkl


#### 1B - OASIS for Clinical utility data split

Note that the data splitting logic for utility and reidentification tasks are different. In Re-identification tasks, we perform a slice level split to intentionally introduce a subject level leakage. For Re-Identification task this becomes a worst-case, and an upper-boundary stress test.

However, for the utility tasks, we perform the splitting in the subject-level to avoid any data leakage, so the clinical utility of the models are more realistic. By doin so we ensure that the utility models are trained purely for the dementia classification and not memorizing patients for other identifiers.

In [16]:
# =============================================================================
# UTILITY TASK - DEMENTIA CLASSIFICATION
# =============================================================================
# This section prepares data for the Utility (Dementia vs Non-Dementia) task.
#
# Key differences from Re-ID:
#   - Slice range: 115-145 (informative band for dementia detection)
#   - Labels: Binary (0=Healthy, 1=Demented)
#   - Split: 5-Fold Stratified Group K-Fold at SUBJECT level (no leakage)
# =============================================================================

import json
import re
from collections import Counter
from sklearn.model_selection import StratifiedGroupKFold

# Utility-specific configuration
UTILITY_SEED = 1337
UTILITY_BAND = (115, 145)
N_FOLDS = 5

# Binary label mapping (4-class -> 2-class)
UTILITY_CLASS_MAPPING = {
    "Non Demented": 0,
    "Very mild Dementia": 1,
    "Mild Dementia": 1,
    "Moderate Dementia": 1,
}

# Output paths
UTILITY_DIR = PROCESSED_DIR / 'utility'
UTILITY_DIR.mkdir(parents=True, exist_ok=True)

print("Utility Task Configuration:")
print(f"  Seed:        {UTILITY_SEED}")
print(f"  Slice band:  {UTILITY_BAND}")
print(f"  N folds:     {N_FOLDS}")
print(f"  Labels:      0=Healthy, 1=Demented")
print(f"  Output dir:  {UTILITY_DIR}")

Utility Task Configuration:
  Seed:        1337
  Slice band:  (115, 145)
  N folds:     5
  Labels:      0=Healthy, 1=Demented
  Output dir:  data/oasis/processed/utility


In [17]:
# =============================================================================
# SCAN FILES FOR UTILITY TASK
# =============================================================================

def scan_utility_files(data_dir: Path) -> list:
    """
    Scan directory and build records with binary labels.
    
    Returns:
        List of dicts with: path, subject_id, slice_num, class_name, label
    """
    print("\n[UTILITY SCAN] Scanning image files...")
    
    records = []
    
    for class_name, label in UTILITY_CLASS_MAPPING.items():
        class_dir = data_dir / class_name
        if not class_dir.exists():
            print(f"  Warning: {class_dir} not found")
            continue
        
        files = list(class_dir.glob("*.jpg"))
        print(f"  {class_name}: {len(files)} files")
        
        for filepath in files:
            filename = filepath.name
            
            # Extract subject ID (e.g., OAS1_0001)
            subj_match = re.match(r'(OAS\d+_\d+)', filename)
            if not subj_match:
                continue
            subject_id = subj_match.group(1)
            
            # Extract slice number
            slice_num = extract_slice_number(filename)
            if slice_num is None:
                continue
            
            records.append({
                'path': str(filepath),
                'subject_id': subject_id,
                'slice_num': slice_num,
                'class_name': class_name,
                'label': label,
            })
    
    print(f"  ✓ Total records: {len(records)}")
    return records


def filter_utility_band(records: list, band: tuple) -> list:
    """Filter records to specified slice band."""
    lo, hi = band
    filtered = [r for r in records if lo <= r['slice_num'] <= hi]
    print(f"\n[UTILITY FILTER] Band {lo}-{hi}: {len(filtered)} slices (from {len(records)})")
    return filtered

In [18]:
# =============================================================================
# GENERATE 5-FOLD SUBJECT-LEVEL SPLITS
# =============================================================================

def generate_utility_folds(records: list, n_folds: int, seed: int) -> dict:
    """
    Generate stratified subject-level folds.
    
    Uses StratifiedGroupKFold to ensure:
      - Subjects are not split across train/val/test
      - Class balance is maintained in each fold
    
    Returns:
        Dict: fold_0..fold_4, each with train/val/test subject lists
    """
    print(f"\n[UTILITY FOLDS] Generating {n_folds}-fold subject splits (seed={seed})...")
    
    # Build subject-level table
    subject_labels = {}
    for r in records:
        sid = r['subject_id']
        lbl = r['label']
        if sid in subject_labels:
            # Verify consistent labels
            if subject_labels[sid] != lbl:
                raise ValueError(f"Inconsistent labels for subject {sid}")
        else:
            subject_labels[sid] = lbl
    
    subj_ids = np.array(list(subject_labels.keys()))
    subj_y = np.array([subject_labels[s] for s in subj_ids])
    
    print(f"  Subjects: {len(subj_ids)}")
    print(f"  Class distribution: {Counter(subj_y)}")
    
    # Outer fold: train+val vs test
    sgkf_outer = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    X_dummy = np.zeros(len(subj_ids))
    
    folds_dict = {}
    
    for k, (trainval_idx, test_idx) in enumerate(sgkf_outer.split(X_dummy, subj_y, groups=subj_ids)):
        trainval_subj = subj_ids[trainval_idx]
        trainval_y = subj_y[trainval_idx]
        test_subj = subj_ids[test_idx]
        
        # Inner fold: train vs val (use same n_splits for consistency)
        sgkf_inner = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed + 100 + k)
        inner_X = np.zeros(len(trainval_subj))
        
        train_idx_inner, val_idx_inner = next(sgkf_inner.split(inner_X, trainval_y, groups=trainval_subj))
        
        train_subj = trainval_subj[train_idx_inner]
        val_subj = trainval_subj[val_idx_inner]
        
        folds_dict[f"fold_{k}"] = {
            'train': train_subj.tolist(),
            'val': val_subj.tolist(),
            'test': test_subj.tolist(),
        }
        
        print(f"  Fold {k}: train={len(train_subj)}, val={len(val_subj)}, test={len(test_subj)}")
    
    # Verify no overlap
    for k in range(n_folds):
        fold = folds_dict[f"fold_{k}"]
        train_set = set(fold['train'])
        val_set = set(fold['val'])
        test_set = set(fold['test'])
        
        assert len(train_set & val_set) == 0, f"Fold {k}: train-val overlap"
        assert len(train_set & test_set) == 0, f"Fold {k}: train-test overlap"
        assert len(val_set & test_set) == 0, f"Fold {k}: val-test overlap"
    
    print("  ✓ No subject overlap across splits")
    
    return folds_dict

In [19]:
# =============================================================================
# PREPARE UTILITY DATASET
# =============================================================================

def prepare_utility_dataset(records: list, folds: dict, output_dir: Path) -> None:
    """
    Prepare and save utility dataset with fold information.
    
    Saves:
      - band_data.pkl: All preprocessed images with metadata
      - folds.json: Subject-level fold assignments
      - folds.pkl: Same as JSON but in pickle format
      - metadata.pkl: Dataset metadata
    """
    print("\n[UTILITY PREP] Preprocessing images...")
    
    # Preprocess all images
    all_images = []
    all_labels = []
    all_subjects = []
    all_slices = []
    
    for r in tqdm(records, desc="  Processing"):
        try:
            img = load_and_preprocess_image(r['path'])
            all_images.append(img)
            all_labels.append(r['label'])
            all_subjects.append(r['subject_id'])
            all_slices.append(r['slice_num'])
        except Exception as e:
            print(f"  ⚠ Skipped {r['path']}: {e}")
    
    # Convert to arrays
    images_array = np.stack(all_images, axis=0)
    labels_array = np.array(all_labels, dtype=np.int64)
    
    print(f"  ✓ Processed {len(images_array)} images")
    print(f"  ✓ Shape: {images_array.shape}")
    print(f"  ✓ Labels: {Counter(labels_array)}")
    
    # Save data bundle
    data_bundle = {
        'images': images_array,
        'labels': labels_array,
        'subjects': all_subjects,
        'slice_nums': all_slices,
        'band': UTILITY_BAND,
        'image_size': IMAGE_SIZE,
    }
    
    save_pickle(data_bundle, output_dir / 'band_data.pkl')
    
    # Save folds
    with open(output_dir / 'folds.json', 'w') as f:
        json.dump(folds, f, indent=2)
    print(f"  Saved: {output_dir / 'folds.json'}")
    
    save_pickle(folds, output_dir / 'folds.pkl')
    
    # Save metadata
    metadata = {
        'task': 'utility',
        'split_type': 'subject_level_5fold',
        'band': UTILITY_BAND,
        'n_folds': N_FOLDS,
        'seed': UTILITY_SEED,
        'num_samples': len(images_array),
        'num_subjects': len(set(all_subjects)),
        'num_classes': 2,
        'class_mapping': UTILITY_CLASS_MAPPING,
        'image_size': IMAGE_SIZE,
    }
    save_pickle(metadata, output_dir / 'metadata.pkl')

In [20]:
# =============================================================================
# VALIDATE UTILITY DATASET
# =============================================================================

def validate_utility_dataset(output_dir: Path) -> bool:
    """Verify utility dataset integrity."""
    print("\n[UTILITY VALIDATE] Checking dataset...")
    
    # Load data
    data = load_pickle(output_dir / 'band_data.pkl')
    folds = load_pickle(output_dir / 'folds.pkl')
    
    images = data['images']
    labels = data['labels']
    subjects = data['subjects']
    
    # Check 1: Image shape
    if images.shape[1:] == IMAGE_SIZE:
        print(f"  ✓ Image shape: {images.shape}")
    else:
        print(f"  ✗ Wrong shape: {images.shape}")
        return False
    
    # Check 2: Labels are binary
    unique_labels = set(labels)
    if unique_labels == {0, 1}:
        print(f"  ✓ Binary labels: {Counter(labels)}")
    else:
        print(f"  ✗ Invalid labels: {unique_labels}")
        return False
    
    # Check 3: Folds have no subject overlap
    all_subjects_set = set(subjects)
    for k in range(N_FOLDS):
        fold = folds[f"fold_{k}"]
        train_set = set(fold['train'])
        val_set = set(fold['val'])
        test_set = set(fold['test'])
        
        # No overlap
        if train_set & val_set or train_set & test_set or val_set & test_set:
            print(f"  ✗ Fold {k}: Subject overlap detected")
            return False
        
        # All fold subjects exist in data
        fold_subjects = train_set | val_set | test_set
        if not fold_subjects.issubset(all_subjects_set):
            print(f"  ✗ Fold {k}: Unknown subjects in fold")
            return False
    
    print(f"  ✓ All {N_FOLDS} folds validated (no subject overlap)")
    
    # Check 4: Class balance per fold
    print("\n  Class balance per fold (subjects):")
    subject_to_label = {s: l for s, l in zip(subjects, labels)}
    # Deduplicate (same subject has same label)
    subject_to_label = {s: subject_to_label[s] for s in set(subjects)}
    
    for k in range(N_FOLDS):
        fold = folds[f"fold_{k}"]
        for split_name in ['train', 'val', 'test']:
            split_subjects = fold[split_name]
            split_labels = [subject_to_label[s] for s in split_subjects]
            counts = Counter(split_labels)
            print(f"    Fold {k} {split_name}: {dict(counts)}")
    
    return True

In [21]:
# =============================================================================
# EXECUTE UTILITY DATA PREPARATION
# =============================================================================

# Step 1: Find data directory
data_dir = RAW_DATA_DIR / 'Data' if (RAW_DATA_DIR / 'Data').exists() else RAW_DATA_DIR

# Step 2: Scan files with utility labels
utility_records = scan_utility_files(data_dir)

# Step 3: Filter to informative band
utility_band_records = filter_utility_band(utility_records, UTILITY_BAND)

# Step 4: Generate 5-fold subject-level splits
utility_folds = generate_utility_folds(utility_band_records, N_FOLDS, UTILITY_SEED)

# Step 5: Prepare and save dataset
prepare_utility_dataset(utility_band_records, utility_folds, UTILITY_DIR)

# Step 6: Validate
validate_utility_dataset(UTILITY_DIR)

print("\n" + "="*60)
print("✓ UTILITY DATA PREPARATION COMPLETE")
print("="*60)
print(f"\nOutput: {UTILITY_DIR}/")
print(f"  ├── band_data.pkl   (images, labels, subjects, slice_nums)")
print(f"  ├── folds.json      (5-fold subject assignments)")
print(f"  ├── folds.pkl       (same as JSON)")
print(f"  └── metadata.pkl    (dataset info)")


[UTILITY SCAN] Scanning image files...
  Non Demented: 67222 files
  Very mild Dementia: 13725 files
  Mild Dementia: 5002 files
  Moderate Dementia: 488 files
  ✓ Total records: 86437

[UTILITY FILTER] Band 115-145: 43927 slices (from 86437)

[UTILITY FOLDS] Generating 5-fold subject splits (seed=1337)...
  Subjects: 347
  Class distribution: Counter({np.int64(0): 266, np.int64(1): 81})
  Fold 0: train=221, val=56, test=70
  Fold 1: train=221, val=56, test=70
  Fold 2: train=222, val=56, test=69
  Fold 3: train=222, val=56, test=69
  Fold 4: train=222, val=56, test=69
  ✓ No subject overlap across splits

[UTILITY PREP] Preprocessing images...


  Processing: 100%|██████████| 43927/43927 [01:26<00:00, 510.74it/s]


  ✓ Processed 43927 images
  ✓ Shape: (43927, 224, 224)
  ✓ Labels: Counter({np.int64(0): 34162, np.int64(1): 9765})
  Saved: data/oasis/processed/utility/band_data.pkl
  Saved: data/oasis/processed/utility/folds.json
  Saved: data/oasis/processed/utility/folds.pkl
  Saved: data/oasis/processed/utility/metadata.pkl

[UTILITY VALIDATE] Checking dataset...
  ✓ Image shape: (43927, 224, 224)
  ✓ Binary labels: Counter({np.int64(0): 34162, np.int64(1): 9765})
  ✓ All 5 folds validated (no subject overlap)

  Class balance per fold (subjects):
    Fold 0 train: {np.int64(0): 168, np.int64(1): 53}
    Fold 0 val: {np.int64(0): 41, np.int64(1): 15}
    Fold 0 test: {np.int64(0): 57, np.int64(1): 13}
    Fold 1 train: {np.int64(0): 169, np.int64(1): 52}
    Fold 1 val: {np.int64(0): 43, np.int64(1): 13}
    Fold 1 test: {np.int64(0): 54, np.int64(1): 16}
    Fold 2 train: {np.int64(0): 171, np.int64(1): 51}
    Fold 2 val: {np.int64(0): 45, np.int64(1): 11}
    Fold 2 test: {np.int64(0): 50, n

### 2. Cheng et. al. Brain Tumor MRI Dataset

In [1]:
# =============================================================================
# DROP-IN REPLACEMENT (CONFIG + DEPENDENCIES) — Cheng data prep
# - Keeps new clean paths: ./data/cheng/raw, ./data/cheng/processed
# - Keeps OASIS-style naming/layout
# - Adds SAFE overwrite toggle for processed artifacts
# - Removes pip-install side effects (assumes mat73 is installed in env)
# =============================================================================

import os
import json
import pickle
import zipfile
import shutil
import re
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Any
from collections import Counter

import numpy as np
import pandas as pd
import requests
from PIL import Image
from tqdm import tqdm

# -----------------------------------------------------------------------------
# CONFIGURATION
# -----------------------------------------------------------------------------

PROJECT_ROOT  = Path(".")
RAW_DATA_DIR  = PROJECT_ROOT / "data" / "cheng" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "cheng" / "processed"

FIGSHARE_ARTICLE_ID = 1512427
IMAGE_SIZE = (224, 224)

# Seeds must match ORIGINAL experiments
REID_SPLIT_SEED    = 42
REID_N_FOLDS       = 5
REID_N_TRAIN_CORE_PER_PID = 4
REID_N_VAL_PER_PID        = 1
REID_N_TEST_PER_PID       = 2
REID_MIN_SLICES_PER_PID   = REID_N_TRAIN_CORE_PER_PID + REID_N_VAL_PER_PID + REID_N_TEST_PER_PID  # 7

UTILITY_SPLIT_SEED = 42
UTILITY_TRAIN_FRAC = 0.70
UTILITY_VAL_FRAC   = 0.15
UTILITY_TEST_FRAC  = 0.15

TUMOR_LABEL_MAP = {1: "meningioma", 2: "glioma", 3: "pituitary"}

# -----------------------------------------------------------------------------
# OVERWRITE CONTROL (requested)
# -----------------------------------------------------------------------------
OVERWRITE_PROCESSED = True  # set False to keep existing ./data/cheng/processed artifacts

if OVERWRITE_PROCESSED:
    for p in [
        PROCESSED_DIR / "reid",
        PROCESSED_DIR / "utility",
        PROCESSED_DIR / "metadata.csv",
    ]:
        if p.exists():
            if p.is_dir():
                shutil.rmtree(p)
            else:
                p.unlink()

# Ensure dirs exist
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"  Raw data:             {RAW_DATA_DIR}")
print(f"  Processed:            {PROCESSED_DIR}")
print(f"  Image size:           {IMAGE_SIZE}")
print(f"  ReID policy:          {REID_N_TRAIN_CORE_PER_PID} train_core + {REID_N_VAL_PER_PID} val + {REID_N_TEST_PER_PID} test, extras->{REID_N_FOLDS} folds")
print(f"  ReID seed:            {REID_SPLIT_SEED}")
print(f"  Utility policy:       subject-level stratified {UTILITY_TRAIN_FRAC}/{UTILITY_VAL_FRAC}/{UTILITY_TEST_FRAC}")
print(f"  Utility seed:         {UTILITY_SPLIT_SEED}")
print(f"  Overwrite processed:  {OVERWRITE_PROCESSED}")

# -----------------------------------------------------------------------------
# DEPENDENCY CHECK (no pip install in notebook)
# -----------------------------------------------------------------------------
try:
    import mat73
except ImportError as e:
    raise ImportError(
        "mat73 is required but not installed in this environment.\n"
        "Install once in your conda env:  pip install mat73"
    ) from e

print(f"  mat73 import: OK (version: {getattr(mat73, '__version__', 'unknown')})")


Configuration:
  Raw data:             data/cheng/raw
  Processed:            data/cheng/processed
  Image size:           (224, 224)
  ReID policy:          4 train_core + 1 val + 2 test, extras->5 folds
  ReID seed:            42
  Utility policy:       subject-level stratified 0.7/0.15/0.15
  Utility seed:         42
  Overwrite processed:  True
  mat73 import: OK (version: unknown)


In [2]:
# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def save_pickle(data: Any, path: Path) -> None:
    """Save data to pickle file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'wb') as f:
        pickle.dump(f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"  Saved: {path}")


def load_pickle(path: Path) -> Any:
    """Load data from pickle file."""
    with open(path, 'rb') as f:
        return pickle.load(f)


def resize_image(img: np.ndarray, target_size: Tuple[int, int]) -> np.ndarray:
    """Resize image to target size using bilinear interpolation."""
    h, w = target_size
    pil_img = Image.fromarray(img.astype(np.float32), mode='F')
    pil_img = pil_img.resize((w, h), resample=Image.BILINEAR)
    return np.array(pil_img, dtype=np.float32)


def per_image_minmax_normalize(images: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """Normalize each image to [0, 1] using its own min/max."""
    x_min = images.reshape(images.shape[0], -1).min(axis=1).reshape(-1, 1, 1)
    x_max = images.reshape(images.shape[0], -1).max(axis=1).reshape(-1, 1, 1)
    return (images - x_min) / (x_max - x_min + eps)

In [3]:
# =============================================================================
# DOWNLOAD FROM FIGSHARE
# =============================================================================

def figshare_get_article(article_id: int) -> Dict[str, Any]:
    """Fetch article metadata from Figshare API."""
    url = f"https://api.figshare.com/v2/articles/{article_id}"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    return r.json()


def download_file(url: str, out_path: Path, chunk_size: int = 1024 * 1024) -> None:
    """Download file with progress indication."""
    if out_path.exists() and out_path.stat().st_size > 0:
        print(f"  ✓ Already exists: {out_path.name}")
        return
    
    print(f"  ⬇️  Downloading: {out_path.name}")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(out_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)


def extract_zip(zip_path: Path, out_dir: Path) -> None:
    """Extract zip file."""
    print(f"  📦 Extracting: {zip_path.name}")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(out_dir)


def download_cheng_dataset() -> Path:
    """Download Cheng Brain Tumor dataset from Figshare."""
    download_dir = RAW_DATA_DIR / 'downloads'
    extract_dir = RAW_DATA_DIR / 'extracted'
    
    # Check if already extracted
    mat_files = list(extract_dir.rglob('*.mat')) if extract_dir.exists() else []
    if len(mat_files) > 100:  # Dataset has ~3000 .mat files
        print(f"✓ Dataset already extracted: {len(mat_files)} .mat files found")
        return extract_dir
    
    download_dir.mkdir(parents=True, exist_ok=True)
    extract_dir.mkdir(parents=True, exist_ok=True)
    
    # Get article info
    print("\n[DOWNLOAD] Fetching Figshare article info...")
    article = figshare_get_article(FIGSHARE_ARTICLE_ID)
    print(f"  Title: {article.get('title', 'Unknown')}")
    
    files = article.get('files', [])
    if not files:
        raise RuntimeError("No files found in Figshare article")
    
    # Download all files
    print(f"\n[DOWNLOAD] Downloading {len(files)} files...")
    for f in files:
        name = f.get('name')
        url = f.get('download_url')
        if name and url:
            out_path = download_dir / name
            download_file(url, out_path)
            
            # Extract if zip
            if out_path.suffix.lower() == '.zip':
                extract_zip(out_path, extract_dir)
    
    mat_files = list(extract_dir.rglob('*.mat'))
    print(f"\n✓ Download complete: {len(mat_files)} .mat files extracted")
    
    return extract_dir

In [4]:
# =============================================================================
# PARSE .MAT FILES (IDENTITY-SAFE, ORIGINAL-COMPATIBLE)
# =============================================================================

def parse_cheng_mat_files(extract_dir: Path) -> Tuple[np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    """
    Parse Cheng Brain Tumor .mat files with ORIGINAL ordering semantics.

    Guarantees:
    - Deterministic global slice order
    - filename preserved exactly
    - metadata.index matches original enumeration
    - PID normalization identical to original
    """

    import mat73

    print("\n[PARSE] Scanning for .mat files...")

    # IMPORTANT: deterministic, filename-based global ordering (same as original)
    mat_files = sorted(
        [p for p in extract_dir.rglob("*.mat") if p.name.lower() != "cvind.mat"],
        key=lambda p: p.name
    )

    print(f"  Found {len(mat_files)} slice .mat files")

    records = []
    failed = 0

    for global_idx, p in enumerate(tqdm(mat_files, desc="  Parsing")):
        try:
            d = mat73.loadmat(str(p))
            cj = d["cjdata"]

            # ---- PID (original-safe) ----
            pid = cj.get("PID")
            if isinstance(pid, str):
                pid = pid.strip()
            else:
                pid = str(np.array(pid).reshape(-1)[0]).strip()

            # ---- Label ----
            label = int(float(np.array(cj.get("label")).reshape(-1)[0]))

            # ---- Image ----
            img_raw = np.array(cj.get("image"), dtype=np.float32)
            img_resized = resize_image(img_raw, IMAGE_SIZE)

            # ---- filename_num (for ReID stress compatibility) ----
            try:
                filename_num = int(p.name.replace(".mat", ""))
            except Exception:
                filename_num = -1  # fallback, matches original behavior

            records.append({
                "index": global_idx,          # CRITICAL: frozen global index
                "filename": p.name,           # CRITICAL: exact filename
                "filename_num": filename_num, # CRITICAL: ReID ordering proxy
                "path": str(p),
                "pid": pid,
                "label": label,
                "image": img_resized,
            })

        except Exception as e:
            failed += 1
            if failed <= 5:
                print(f"  ⚠ Failed: {p.name} -> {e}")

    print(f"  ✓ Parsed: {len(records)} | Failed: {failed}")
    if not records:
        raise RuntimeError("No records parsed!")

    # ---- Stack arrays in EXACT parsed order ----
    images = np.stack([r["image"] for r in records]).astype(np.float32)
    labels = np.array([r["label"] for r in records], dtype=np.int32)
    pids   = np.array([r["pid"]   for r in records], dtype=object)

    print("  Normalizing images (per-image min-max)...")
    images = per_image_minmax_normalize(images)

    # ---- Metadata (index is authoritative) ----
    metadata = pd.DataFrame([{
        "index": r["index"],
        "filename": r["filename"],
        "filename_num": r["filename_num"],
        "path": r["path"],
        "pid": r["pid"],
        "label": r["label"],
    } for r in records])

    # Sanity
    assert np.all(metadata["index"].values == np.arange(len(metadata))), \
        "Global index mismatch — ordering corruption detected"

    print(f"\n  Images: {images.shape}, dtype={images.dtype}")
    print(f"  Labels: {Counter(labels)}")
    print(f"  Unique PIDs: {len(np.unique(pids))}")

    return images, labels, pids, metadata


In [5]:
# =============================================================================
# RE-ID SPLIT (Slice-Level)
# =============================================================================

def build_reid_split(
    pids: np.ndarray,
    seed: int = None,   # kept for API compatibility; NOT used
    n_folds: int = 5,
    n_train_core: int = 4,
    n_val: int = 1,
    n_test: int = 2,
    min_slices: int = 7,
) -> Dict[str, Any]:
    """
    Deterministic Re-ID split matching the ORIGINAL Cheng stress-test logic.

    For each PID with >= min_slices:
      - Use deterministic ordering (global index order, already filename-sorted upstream)
      - test      = LAST  n_test slices
      - val       = slices just before train_core
      - train_core= FIRST n_train_core slices
      - extras    = remaining middle slices, round-robin into folds

    NOTE:
      - No RNG
      - `seed` is metadata-only (kept to match newer APIs)
    """

    print("\n[RE-ID SPLIT] Building deterministic split (NO RNG)")
    print(f"  Per-PID allocation: {n_train_core} train + {n_val} val + {n_test} test")
    print(f"  Min slices required: {min_slices}")

    pids = np.asarray(pids).astype(str)

    # PID -> global indices (already in deterministic order)
    pid_to_indices: Dict[str, List[int]] = {}
    for i, pid in enumerate(pids):
        pid_to_indices.setdefault(pid, []).append(int(i))

    kept_pids = []
    excluded_pids = []
    per_pid = {}

    train_core_all = []
    val_all = []
    test_all = []
    extras_all = []

    fold_extras = {f"fold_{k}": [] for k in range(n_folds)}

    for pid, idxs in pid_to_indices.items():
        idxs = np.array(idxs, dtype=int)

        if len(idxs) < min_slices:
            excluded_pids.append(pid)
            continue

        kept_pids.append(pid)

        # Deterministic slicing (NO SHUFFLE)
        test_idx = idxs[-n_test:]
        val_idx = idxs[-(n_test + n_val):-n_test]
        train_core_idx = idxs[:n_train_core]
        extra_idx = idxs[n_train_core:-(n_test + n_val)]

        # Distribute extras round-robin into folds
        for j, gidx in enumerate(extra_idx.tolist()):
            fk = f"fold_{j % n_folds}"
            fold_extras[fk].append(int(gidx))

        per_pid[pid] = {
            "n_slices": int(len(idxs)),
            "train_core_idx": train_core_idx.tolist(),
            "val_idx": val_idx.tolist(),
            "test_idx": test_idx.tolist(),
            "extra_idx": extra_idx.tolist(),
        }

        train_core_all.extend(train_core_idx.tolist())
        val_all.extend(val_idx.tolist())
        test_all.extend(test_idx.tolist())
        extras_all.extend(extra_idx.tolist())

    # Global sets (sorted for stability)
    train_core_all = sorted(train_core_all)
    val_all = sorted(val_all)
    test_all = sorted(test_all)
    extras_all = sorted(extras_all)

    eligible_idx = sorted(set(train_core_all + val_all + test_all + extras_all))

    # Sanity checks
    assert not set(train_core_all) & set(val_all)
    assert not set(train_core_all) & set(test_all)
    assert not set(val_all) & set(test_all)

    print(f"  ✓ Kept PIDs: {len(kept_pids)} | Excluded: {len(excluded_pids)}")
    print(f"  ✓ Train core: {len(train_core_all)} | Val: {len(val_all)} | Test: {len(test_all)} | Extras: {len(extras_all)}")

    return {
        "_meta": {
            "ordering": "deterministic (global index order)",
            "seed": None,
            "n_folds": n_folds,
            "min_slices_per_pid": min_slices,
            "n_train_core_per_pid": n_train_core,
            "n_val_per_pid": n_val,
            "n_test_per_pid": n_test,
            "n_pids_total": len(pid_to_indices),
            "n_pids_kept": len(kept_pids),
            "n_pids_excluded": len(excluded_pids),
            "timestamp": datetime.now().isoformat(),
        },
        "kept_pids": kept_pids,
        "excluded_pids": excluded_pids,
        "per_pid": per_pid,
        "global": {
            "train_core_idx": train_core_all,
            "val_idx": val_all,
            "test_idx": test_all,
            "extra_idx": extras_all,
            "eligible_idx": eligible_idx,
        },
        "folds": {
            f"fold_{k}": {
                "extra_train_idx": sorted(set(fold_extras[f"fold_{k}"]))
            }
            for k in range(n_folds)
        },
    }


In [7]:
# =============================================================================
# UTILITY SPLIT (Subject-Level, Stratified)
# =============================================================================

def build_utility_split(
    pids: np.ndarray,
    labels: np.ndarray,
    seed: int,
    train_frac: float = 0.70,
    val_frac: float = 0.15,
    test_frac: float = 0.15,
) -> Dict[str, Any]:
    """
    Build utility split with subject-level separation.
    Matches ORIGINAL Cheng clinical utility split logic.
    """

    print(f"\n[UTILITY SPLIT] Building split (seed={seed})...")
    print(f"  Fractions: train={train_frac}, val={val_frac}, test={test_frac}")

    rng = np.random.default_rng(seed)
    pids = np.asarray(pids).astype(str)
    labels = np.asarray(labels)

    # Build PID → label table (deterministic due to preserved metadata order)
    pid_to_label = {}
    for pid, lbl in zip(pids, labels):
        if pid in pid_to_label:
            assert pid_to_label[pid] == lbl, f"Inconsistent labels for PID {pid}"
        else:
            pid_to_label[pid] = int(lbl)

    patient_df = pd.DataFrame(
        [{"pid": pid, "label": lbl} for pid, lbl in pid_to_label.items()]
    )

    print(f"  Total patients: {len(patient_df)}")
    print(f"  Class distribution: {dict(patient_df['label'].value_counts().sort_index())}")

    train_pids, val_pids, test_pids = [], [], []

    # Stratified by label (same as original)
    for lbl, group in patient_df.groupby("label", sort=True):
        pids_lbl = group["pid"].to_numpy(object)
        rng.shuffle(pids_lbl)

        n = len(pids_lbl)
        n_train = int(round(n * train_frac))
        n_val = int(round(n * val_frac))
        n_test = n - n_train - n_val  # remainder to test

        train_pids.extend(pids_lbl[:n_train].tolist())
        val_pids.extend(pids_lbl[n_train:n_train + n_val].tolist())
        test_pids.extend(pids_lbl[n_train + n_val:].tolist())

    # Safety checks
    assert not set(train_pids) & set(val_pids)
    assert not set(train_pids) & set(test_pids)
    assert not set(val_pids) & set(test_pids)

    print(f"  ✓ Train PIDs: {len(train_pids)} | Val: {len(val_pids)} | Test: {len(test_pids)}")

    pid_arr = np.asarray(pids).astype(str)
    train_idx = np.where(np.isin(pid_arr, train_pids))[0].tolist()
    val_idx   = np.where(np.isin(pid_arr, val_pids))[0].tolist()
    test_idx  = np.where(np.isin(pid_arr, test_pids))[0].tolist()

    print(f"  ✓ Train slices: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

    return {
        "_meta": {
            "seed": int(seed),
            "train_frac": train_frac,
            "val_frac": val_frac,
            "test_frac": test_frac,
            "n_pids_total": len(patient_df),
            "n_pids_train": len(train_pids),
            "n_pids_val": len(val_pids),
            "n_pids_test": len(test_pids),
            "timestamp": datetime.now().isoformat(),
        },
        "train_pids": train_pids,
        "val_pids": val_pids,
        "test_pids": test_pids,
        "train_idx": train_idx,
        "val_idx": val_idx,
        "test_idx": test_idx,
    }


In [8]:
# =============================================================================
# PREPARE AND SAVE DATASETS
# =============================================================================

def prepare_reid_dataset(
    images: np.ndarray,
    pids: np.ndarray,
    split: Dict,
    output_dir: Path
) -> None:
    """Save Re-ID dataset with split information (original-compatible)."""
    print("\n[RE-ID SAVE] Saving Re-ID dataset...")

    reid_dir = output_dir / "reid"
    reid_dir.mkdir(parents=True, exist_ok=True)

    # PID → class mapping (stable, sorted)
    kept_pids = sorted(split["kept_pids"])
    pid_to_class = {pid: i for i, pid in enumerate(kept_pids)}

    # Data bundle (ALL slices preserved)
    data_bundle = {
        "images": images.astype(np.float32, copy=False),
        "pids": pids.astype(object, copy=False),
        "pid_to_class": pid_to_class,
        "num_classes": len(kept_pids),
        "image_size": IMAGE_SIZE,
    }

    with open(reid_dir / "data_bundle.pkl", "wb") as f:
        pickle.dump(data_bundle, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"  Saved: {reid_dir / 'data_bundle.pkl'}")

    # Split
    with open(reid_dir / "split.json", "w") as f:
        json.dump(split, f, indent=2)
    print(f"  Saved: {reid_dir / 'split.json'}")

    with open(reid_dir / "split.pkl", "wb") as f:
        pickle.dump(split, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"  Saved: {reid_dir / 'split.pkl'}")

    # Metadata (NO assumed seed)
    metadata = {
        "task": "reid",
        "split_type": "slice_level_with_folds",
        "num_classes": len(kept_pids),
        "n_pids_kept": len(kept_pids),
        "n_pids_excluded": len(split["excluded_pids"]),
        "image_size": IMAGE_SIZE,
        "ordering": split["_meta"].get("ordering", "unknown"),
    }

    with open(reid_dir / "metadata.pkl", "wb") as f:
        pickle.dump(metadata, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"  Saved: {reid_dir / 'metadata.pkl'}")


def prepare_utility_dataset(
    images: np.ndarray,
    labels: np.ndarray,
    pids: np.ndarray,
    split: Dict,
    output_dir: Path
) -> None:
    """Save Utility dataset with subject-level split."""
    print("\n[UTILITY SAVE] Saving Utility dataset...")

    utility_dir = output_dir / "utility"
    utility_dir.mkdir(parents=True, exist_ok=True)

    # Data bundle (ALL slices preserved)
    data_bundle = {
        "images": images.astype(np.float32, copy=False),
        "labels": labels.astype(np.int32, copy=False),
        "pids": pids.astype(object, copy=False),
        "label_map": TUMOR_LABEL_MAP,
        "num_classes": len(TUMOR_LABEL_MAP),
        "image_size": IMAGE_SIZE,
    }

    with open(utility_dir / "data_bundle.pkl", "wb") as f:
        pickle.dump(data_bundle, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"  Saved: {utility_dir / 'data_bundle.pkl'}")

    # Split
    with open(utility_dir / "split.json", "w") as f:
        json.dump(split, f, indent=2)
    print(f"  Saved: {utility_dir / 'split.json'}")

    with open(utility_dir / "split.pkl", "wb") as f:
        pickle.dump(split, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"  Saved: {utility_dir / 'split.pkl'}")

    # Metadata
    metadata = {
        "task": "utility",
        "split_type": "subject_level",
        "num_classes": len(TUMOR_LABEL_MAP),
        "label_map": TUMOR_LABEL_MAP,
        "image_size": IMAGE_SIZE,
        "seed": split["_meta"]["seed"],
    }

    with open(utility_dir / "metadata.pkl", "wb") as f:
        pickle.dump(metadata, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"  Saved: {utility_dir / 'metadata.pkl'}")


In [9]:
# =============================================================================
# VALIDATION
# =============================================================================

def validate_reid_dataset(output_dir: Path) -> bool:
    """Validate Re-ID dataset."""
    print("\n[VALIDATE RE-ID] Checking dataset...")
    
    reid_dir = output_dir / 'reid'
    
    with open(reid_dir / 'data_bundle.pkl', 'rb') as f:
        data = pickle.load(f)
    with open(reid_dir / 'split.pkl', 'rb') as f:
        split = pickle.load(f)
    
    images = data['images']
    pids = data['pids']
    kept_pids = set(split['kept_pids'])
    g = split['global']
    
    # Check 1: Image shape
    if images.shape[1:] == IMAGE_SIZE:
        print(f"  ✓ Image shape: {images.shape}")
    else:
        print(f"  ✗ Wrong shape: {images.shape}")
        return False
    
    # Check 2: No slice overlap
    train_set = set(g['train_core_idx'])
    val_set = set(g['val_idx'])
    test_set = set(g['test_idx'])
    
    if not (train_set & val_set) and not (train_set & test_set) and not (val_set & test_set):
        print("  ✓ No slice overlap between splits")
    else:
        print("  ✗ Slice overlap detected!")
        return False
    
    # Check 3: All PIDs in all splits (closed-set)
    train_pids = {pids[i] for i in g['train_core_idx']}
    val_pids = {pids[i] for i in g['val_idx']}
    test_pids = {pids[i] for i in g['test_idx']}
    
    if train_pids == val_pids == test_pids == kept_pids:
        print(f"  ✓ Closed-set: {len(kept_pids)} PIDs in all splits")
    else:
        print("  ✗ PID mismatch across splits!")
        return False
    
    # Check 4: Folds
    n_folds = split['_meta']['n_folds']
    total_extras = sum(len(split['folds'][f'fold_{k}']['extra_train_idx']) for k in range(n_folds))
    print(f"  ✓ {n_folds} folds with {total_extras} total extra slices")
    
    return True


def validate_utility_dataset(output_dir: Path) -> bool:
    """Validate Utility dataset."""
    print("\n[VALIDATE UTILITY] Checking dataset...")
    
    utility_dir = output_dir / 'utility'
    
    with open(utility_dir / 'data_bundle.pkl', 'rb') as f:
        data = pickle.load(f)
    with open(utility_dir / 'split.pkl', 'rb') as f:
        split = pickle.load(f)
    
    images = data['images']
    pids = data['pids']
    labels = data['labels']
    
    # Check 1: Image shape
    if images.shape[1:] == IMAGE_SIZE:
        print(f"  ✓ Image shape: {images.shape}")
    else:
        print(f"  ✗ Wrong shape: {images.shape}")
        return False
    
    # Check 2: No PID overlap
    train_pids = set(split['train_pids'])
    val_pids = set(split['val_pids'])
    test_pids = set(split['test_pids'])
    
    if not (train_pids & val_pids) and not (train_pids & test_pids) and not (val_pids & test_pids):
        print(f"  ✓ No PID overlap (subject-level split)")
    else:
        print("  ✗ PID overlap detected!")
        return False
    
    # Check 3: Labels are valid
    unique_labels = set(labels)
    if unique_labels == {1, 2, 3}:
        print(f"  ✓ Labels: {Counter(labels)}")
    else:
        print(f"  ✗ Invalid labels: {unique_labels}")
        return False
    
    # Check 4: Class balance per split
    print("\n  Class balance per split (slices):")
    for split_name, idxs in [('train', split['train_idx']), ('val', split['val_idx']), ('test', split['test_idx'])]:
        split_labels = [labels[i] for i in idxs]
        print(f"    {split_name}: {dict(Counter(split_labels))}")
    
    return True

In [10]:
# =============================================================================
# MAIN EXECUTION
# =============================================================================

# Step 1: Download dataset
extract_dir = download_cheng_dataset()

# Step 2: Parse .mat files
images, labels, pids, metadata = parse_cheng_mat_files(extract_dir)

# Step 3: Build Re-ID split
reid_split = build_reid_split(
    pids=pids,
    seed=REID_SPLIT_SEED,
    n_folds=REID_N_FOLDS,
    n_train_core=REID_N_TRAIN_CORE_PER_PID,
    n_val=REID_N_VAL_PER_PID,
    n_test=REID_N_TEST_PER_PID,
    min_slices=REID_MIN_SLICES_PER_PID,
)

# Step 4: Build Utility split
utility_split = build_utility_split(
    pids=pids,
    labels=labels,
    seed=UTILITY_SPLIT_SEED,
    train_frac=UTILITY_TRAIN_FRAC,
    val_frac=UTILITY_VAL_FRAC,
    test_frac=UTILITY_TEST_FRAC,
)

# Step 5: Save datasets
prepare_reid_dataset(images, pids, reid_split, PROCESSED_DIR)
prepare_utility_dataset(images, labels, pids, utility_split, PROCESSED_DIR)

# Step 6: Save metadata CSV
metadata.to_csv(PROCESSED_DIR / 'metadata.csv', index=False)
print(f"\n  Saved: {PROCESSED_DIR / 'metadata.csv'}")

# Step 7: Validate
validate_reid_dataset(PROCESSED_DIR)
validate_utility_dataset(PROCESSED_DIR)

print("\n" + "="*70)
print("✓ CHENG BRAIN TUMOR DATASET PREPARATION COMPLETE")
print("="*70)
print(f"\nOutput: {PROCESSED_DIR}/")
print(f"  ├── reid/")
print(f"  │   ├── data_bundle.pkl  (images, pids, pid_to_class)")
print(f"  │   ├── split.json       (train_core + val + test + fold extras)")
print(f"  │   ├── split.pkl")
print(f"  │   └── metadata.pkl")
print(f"  ├── utility/")
print(f"  │   ├── data_bundle.pkl  (images, labels, pids)")
print(f"  │   ├── split.json       (subject-level 70/15/15)")
print(f"  │   ├── split.pkl")
print(f"  │   └── metadata.pkl")
print(f"  └── metadata.csv")
print("="*70)

✓ Dataset already extracted: 3064 .mat files found

[PARSE] Scanning for .mat files...
  Found 3064 slice .mat files


  Parsing: 100%|██████████| 3064/3064 [00:31<00:00, 98.49it/s] 


  ✓ Parsed: 3064 | Failed: 0
  Normalizing images (per-image min-max)...

  Images: (3064, 224, 224), dtype=float32
  Labels: Counter({np.int32(2): 1426, np.int32(3): 930, np.int32(1): 708})
  Unique PIDs: 233

[RE-ID SPLIT] Building LEGACY-EXACT split (seed ignored, deterministic)...
  ✓ Kept PIDs: 175 | Excluded: 58
  ✓ Train core: 700 | Val: 175 | Test: 350 | Extras: 1631

[UTILITY SPLIT] Building split (seed=42)...
  Fractions: train=0.7, val=0.15, test=0.15
  Total patients: 233
  Class distribution: {1: np.int64(82), 2: np.int64(89), 3: np.int64(62)}
  ✓ Train PIDs: 162 | Val: 34 | Test: 37
  ✓ Train slices: 2074 | Val: 498 | Test: 492

[RE-ID SAVE] Saving Re-ID dataset...
  Saved: data/cheng/processed/reid/data_bundle.pkl
  Saved: data/cheng/processed/reid/split.json
  Saved: data/cheng/processed/reid/split.pkl
  Saved: data/cheng/processed/reid/metadata.pkl

[UTILITY SAVE] Saving Utility dataset...
  Saved: data/cheng/processed/utility/data_bundle.pkl
  Saved: data/cheng/proces